In [1]:
import pandas as pd
import torch
import numpy as np

In [2]:
#Loading in SIMPA datasets -- original and and (syntactic + semantic) simplified

original = pd.read_table('/home/c23068554/final_project/datasets/simpa/ss.original', header=None)
simplified = pd.read_table('/home/c23068554/final_project/datasets/simpa/ss.simplified', header=None)

#Combine datasets into one dataframe
df = pd.DataFrame({'original': original[0], 'simplified': simplified[0]})

#### Test sizes:
3 - 0.0025,
2 - 0.001,
1 - 0.0001

In [3]:
# data splitting, splitting data values to be tested and some to be used as few-shot
from sklearn.model_selection import train_test_split

random_state = 59
data_test, data_train = train_test_split(df, test_size = 0.0025, random_state = random_state)

print(len(data_test))
print(len(data_train))

1097
3


# Making prompt with randomized n-shot prompting



In [4]:
#Instruction from BLESS prompt 2
instruction = "Please rewrite the following complex sentence in order to make it easier to understand by non-native speakers of English. You can do so by replacing complex words with simpler synonyms (i.e. paraphrasing), deleting unimportant information (i.e. compression), and/or splitting a long complex sentence into several simpler ones. The final simplified sentence needs to be grammatical, fluent, and retain the main ideas of its original counterpart without altering its meaning. Ensure there is a change.\n\n"

def makePrompt(instruction, examples):
  #formatting text for fewshot examples
  fewshot = ""
  for index, row in examples.iterrows():
    fewshot += (f"Complex: {row.loc['original']}\nSimple: {row.loc['simplified']}\n\n")
  return(instruction + fewshot)

fewshot_example = makePrompt(instruction, data_train)
print(fewshot_example)

Please rewrite the following complex sentence in order to make it easier to understand by non-native speakers of English. You can do so by replacing complex words with simpler synonyms (i.e. paraphrasing), deleting unimportant information (i.e. compression), and/or splitting a long complex sentence into several simpler ones. The final simplified sentence needs to be grammatical, fluent, and retain the main ideas of its original counterpart without altering its meaning. Ensure there is a change.

Complex: However, it is advisable to submit your application as soon as practicable to ensure no delay is made should your application be referred to our Licensing Committee.
Simple: It is best to submit your application as soon as possible. This will ensure there is no delay if your application has to be referred to our Licensing Committee.

Complex: If you are not sure who to pay the rent to, you could either carry on paying it to the old landlord or set the money aside in a separate bank acc

# Load in model and inference

In [5]:
# Test for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"compute type: {device}")

compute type: cuda


In [6]:
# Models
flanT5small = "google/flan-t5-small"
flanT5large = "google/flan-t5-large"
flanT5xl = "google/flan-t5-xl" #3B parameters
mT5 = "google/mt5-large" #1.2B parameters 

In [7]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained(flanT5large)
model = AutoModelForSeq2SeqLM.from_pretrained(flanT5large).to(device)

/home/c23068554/miniconda3/envs/venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [8]:
def generateT5(prompt):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    output = model.generate(
        input_ids,
        max_new_tokens = 512,
        #do_sample=False,
        #num_beams=1,
        #encoder_no_repeat_ngram_size=5
    )
    return tokenizer.decode(
        output[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    ).strip()

In [9]:
print(model.generation_config)

GenerationConfig {
  "_from_model_config": true,
  "decoder_start_token_id": 0,
  "eos_token_id": 1,
  "pad_token_id": 0,
  "transformers_version": "4.31.0"
}



In [10]:
#looping through all testing subset
def promptLoop(fewshot_example, data_test):
  LMsimplified = []
  token_counts = []
  for row in data_test['original']:
    full_prompt = fewshot_example + f"Complex: {row}\nSimple: "
    #print(full_prompt)
    token_counts.append(len(tokenizer(full_prompt)['input_ids']))
    LMsimplified.append(generateT5(full_prompt))
  print(token_counts)
  return LMsimplified

In [11]:
LMoutput = (promptLoop(fewshot_example, data_test))

#print(promptLoop(fewshot_example, data_test))

[407, 408, 399, 397, 395, 418, 400, 392, 390, 432, 398, 389, 390, 422, 414, 391, 393, 399, 404, 419, 402, 400, 396, 418, 395, 398, 403, 398, 421, 420, 398, 391, 396, 416, 406, 389, 389, 395, 408, 397, 410, 395, 394, 411, 405, 394, 407, 401, 406, 396, 391, 413, 390, 394, 408, 393, 408, 390, 395, 390, 428, 409, 396, 407, 399, 393, 389, 398, 398, 391, 401, 415, 402, 396, 401, 392, 388, 393, 399, 396, 388, 411, 394, 403, 395, 405, 392, 398, 417, 404, 402, 391, 397, 395, 420, 390, 405, 389, 407, 395, 425, 412, 393, 414, 411, 400, 391, 413, 436, 400, 412, 396, 390, 404, 394, 395, 394, 399, 400, 393, 414, 407, 395, 408, 391, 388, 392, 404, 408, 387, 410, 401, 391, 390, 403, 407, 397, 389, 397, 410, 396, 398, 391, 394, 390, 394, 409, 399, 390, 400, 400, 397, 391, 403, 400, 390, 402, 396, 392, 399, 395, 434, 395, 399, 398, 399, 407, 427, 397, 412, 393, 392, 393, 391, 389, 395, 393, 411, 418, 388, 386, 405, 405, 395, 404, 389, 402, 404, 397, 419, 408, 428, 391, 392, 390, 399, 402, 406, 394, 402,

# Evaluation

### Preliminary tests on the model
 How many sentences dont actually get simplified by the model?

In [12]:
count = 0
for original, simple, lmSimple in zip(data_test['original'], data_test['simplified'], LMoutput):
  print(original)
  print(simple)
  print(lmSimple + "\n")
  if original == lmSimple:
    #print(original)
    #print(simple)
    #print(lmSimple)
    count += 1
print(f"Number of sentences not simplified or altered by the model: {count}")

The exhibitions served a number of purposes - their main focus was to promote business and industry, open up new markets, and generally to outperform competitors, in an increasingly global economic market.
The exhibitions served a number of purposes. The exhibitions main focus was to promote business and industry, open up new markets, and generally to outperform competitors, in an increasingly global economic market
The exhibitions served a number of purposes. Their main focus was to promote business and industry, open up new markets, and generally to outperform competitors, in an increasingly global economic market.

To aid biodiversity conservation we have drawn up Habitat Action Plans (HAPs) for grassland, woodland, heathland and wetland habitats across 130 target sites in Sheffield.
To aid biodiversity conservation we have drawn up Habitat Action Plans (HAPs) for grassland, woodland, heathland and wetland habitats across 130 target sites in Sheffield.
To help with biodiversity cons

In [13]:
import evaluate

simple = data_test['simplified'].tolist()
source = data_test['original'].tolist()

#load metrics
rouge = evaluate.load('rouge')
bleu = evaluate.load('bleu')
bertscore = evaluate.load('bertscore')
sari = evaluate.load('sari')

#Converting simplified reference sentences into a list inside a list
references = [[s] for s in data_test["simplified"].astype(str).tolist()]
sari_score = sari.compute(sources= source, predictions= LMoutput, references= references)

# Compute scores
rouge_results = rouge.compute(predictions=LMoutput, references=simple)
bleu_results = bleu.compute(predictions=LMoutput, references=simple)

bertscore_compute = bertscore.compute(predictions=LMoutput, references=simple, lang='en')
berstcoreAvg = np.mean(bertscore_compute['f1'])

/home/c23068554/miniconda3/envs/venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### FKGL Score

In [14]:
import textstat
textstat.set_lang("en")

originalAvg   = np.mean([textstat.flesch_reading_ease(s) for s in data_test["original"].tolist()])
simplifiedAvg = np.mean([textstat.flesch_reading_ease(s) for s in LMoutput])

### LENS score

In [15]:
from lens import download_model, LENS

lens_path = download_model("davidheineman/lens")
lens = LENS(lens_path, rescale=True)

scores = lens.score(data_test["original"].tolist(), LMoutput, references, devices = [0])
lensAvg = np.mean(scores)

/home/c23068554/miniconda3/envs/venv/lib/python3.10/site-packages/lightning_fabric/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

/home/c23068554/miniconda3/envs/venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Using LENS with topk=3


/home/c23068554/miniconda3/envs/venv/lib/python3.10/site-packages/pytorch_lightning/core/saving.py:165: UserWarning: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
  rank_zero_warn(
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.utilities.rank_zero:You are using a CUDA device ('NVIDIA RTX 6000 Ada Generation') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
INFO:pytorch_l

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Predicting DataLoader 0: 100%|██████████| 69/69 [00:04<00:00, 14.68it/s]


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [16]:
#output
print(f"Dataset: SIMPA, size: {len(data_test)}, random state: {random_state}")
print(f"Model:{flanT5large}")
print(f"Prompt: BLESS 2")
print()
print(f"ROUGE Score: {rouge_results['rouge1']}")
print(f"BLEU Score: {bleu_results['bleu']}")
print(f"BERTScore Score: {berstcoreAvg}")
print(f"Sari Score: {sari_score['sari']}")
print()
print(f"Original Flesch score: {originalAvg}")
print(f"Simplified Flesch score: {simplifiedAvg}")
print(f"Average LENS score: {lensAvg}")
print()
print(f"unsimplified sentences: {count}/{len(data_test)}")

Dataset: SIMPA, size: 1097, random state: 59
Model:google/flan-t5-large
Prompt: BLESS 2

ROUGE Score: 0.7282787180154735
BLEU Score: 0.49378097047492675
BERTScore Score: 0.9533459716421316
Sari Score: 48.47542152651919

Original Flesch score: 38.093855475465475
Simplified Flesch score: 46.52332689918671
Average LENS score: 58.95062816740289

unsimplified sentences: 259/1097


In [24]:
print(f"\nTriple click and paste into first metric column")
print(f"{sari_score['sari']:.4f}\t{rouge_results['rouge1']:.4f}\t{bleu_results['bleu']:.4f}\t{berstcoreAvg:.4f}\t{lensAvg:.4f}\t{simplifiedAvg:.4f}\t{count}/{len(data_test)}")


Triple click and paste into first metric column
48.4754	0.7283	0.4938	0.9533	58.9506	46.5233	259/1097


In [18]:
def normalize_text(s):
    return " ".join(str(s).strip().split())

stripCount = 0
for original, lmSimple in zip(data_test["original"], LMoutput):
    if normalize_text(original) == normalize_text(lmSimple):
        stripCount += 1

print(count)

259


In [19]:
import csv

exp_id = "S-1"  # <-- change per experiment
out_path = f"/tmp/{exp_id}_results.csv"

with open(out_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(["Exp ID", "Index", "Original", "Reference", "LM Output"])
    for i, (orig, ref, lm) in enumerate(zip(source, simple, LMoutput)):
        writer.writerow([exp_id, i, orig, ref, lm])

print(f"Wrote {len(source)} rows to {out_path}")

# Then run: scp user@server:/tmp/{exp_id}_results.csv ~/Desktop/
#           ssh user@server "rm /tmp/{exp_id}_results.csv"

Wrote 1097 rows to /tmp/S-1_results.csv


In [20]:
# command to clear cache often, to reduce disk space used:
# rm -rf ~/.cache/*

#check disk space used:
# du -h --max-depth=1 ~ | sort -h
